# SongoFish -- entrainement sur Google Colab

Remplace GCP Compute Engine comme provider de calcul pour ce projet.
Datasets et checkpoints vivent sur Google Drive (pas sur Colab lui-meme,
qui est ephemere) ; le code vient du depot GitHub `songo-fish`.

Pre-requis : le dossier `SongoFish/` doit deja exister sur le Drive du
compte utilise ici, avec `datasets/` et `checkpoints/` dedans (voir
README du dossier Drive).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/SongoFish'  # ajuster si le dossier partage a un autre chemin
import os
assert os.path.isdir(DRIVE_ROOT), f"Dossier introuvable : {DRIVE_ROOT} -- verifier le montage/partage Drive"
print('Contenu :', os.listdir(DRIVE_ROOT))

In [ ]:
# Code du projet : clone direct depuis GitHub (depot prive -> demande un
# token d'acces personnel si `git clone` anonyme echoue)
!git clone https://github.com/GlennEriss/songo-fish.git /content/songo-fish
%cd /content/songo-fish
!pip install -q -e ".[train]"

In [ ]:
# Verifie GPU/CPU -- rappel du projet (session locale) : ce reseau est si
# petit que le GPU n'apporte souvent aucun gain (voire ralentit), a
# mesurer ici avant de forcer device='cuda'
import torch
print('CUDA disponible:', torch.cuda.is_available())

In [ ]:
from pathlib import Path
from songo_ai.model import train_and_register, promote_version, get_champion

DATASET_DIR = Path(DRIVE_ROOT) / 'datasets' / 'dataset_v005_400k'  # ajuster au dataset vise
CHECKPOINT_DIR = Path(DRIVE_ROOT) / 'checkpoints'
REGISTRY_PATH = CHECKPOINT_DIR / 'registry.json'

VERSION = '0.2.1'  # nouvelle version : plafond d'epoques releve (150) suite
                    # a la mesure locale montrant que l'entrainement n'avait
                    # pas fini de converger a 60 epoques

manifest = train_and_register(
    version=VERSION,
    train_shard=DATASET_DIR / 'train.jsonl',
    val_shard=DATASET_DIR / 'val.jsonl',
    dataset_manifest_path=DATASET_DIR / 'manifest.json',
    checkpoint_dir=CHECKPOINT_DIR,
    registry_path=REGISTRY_PATH,
    epochs=150,
    early_stopping_patience=15,
    device='cpu',
    notes='Entraine sur Google Colab (migration depuis GCP Compute Engine)',
)
print(f"Version {manifest.version} entrainee")
print(f"epoques={manifest.metrics['epochs_run']} meilleure_epoque={manifest.metrics['best_epoch']}")
print(f"val_loss={manifest.metrics['val_loss']:.4f} val_top1={manifest.metrics['val_policy_top1']:.3f}")

In [ ]:
# Tournoi vs champion actuel avant toute promotion (jamais automatique,
# cf. apps/trainer/README.md)
from songo_ai.evaluation import play_match
from songo_ai.model import load_model, make_network_agent

OPENING_RANDOM_PLIES = 6
TOURNAMENT_GAMES = 200

checkpoint_path = CHECKPOINT_DIR / f'model_v{VERSION}.pt'
challenger = make_network_agent(load_model(checkpoint_path, dropout=manifest.architecture['dropout']), temperature=0.15)

champion = get_champion(registry_path=REGISTRY_PATH)
if champion is not None and champion['version'] != VERSION:
    champion_agent = make_network_agent(
        load_model(Path(champion['checkpoint_path']), dropout=champion['architecture']['dropout']), temperature=0.15
    )
    result = play_match(challenger, champion_agent, num_games=TOURNAMENT_GAMES,
                         seed=hash(VERSION) % 10_000, opening_random_plies=OPENING_RANDOM_PLIES)
    print(f"challenger v{VERSION} vs champion v{champion['version']}: "
          f"score={result.score_a:.1f}/{result.games} ({result.win_rate_a*100:.1f}%, "
          f"IC95 [{result.ci_low*100:.1f}%, {result.ci_high*100:.1f}%])")
    print('Promotion : appeler promote_version(...) explicitement apres revue -- jamais automatique.')

In [ ]:
# Promotion explicite (decommenter apres revue des resultats ci-dessus) :
# promote_version(VERSION, registry_path=REGISTRY_PATH)